# NEON 조사구 전체 CHM 높이 정보로 AGB 회귀 — 작은 비교 실험 1개

2026-11-12 · **미실행 Colab 노트북** · CPU 런타임으로 충분합니다.

나무를 개별 탐지하지 않고 조사구의 평균 CHM 높이와 2m 이상 피복률만으로
지상부 바이오매스 **AGB (Mg/ha = t/ha)** 를 추정합니다. 탄소(tC/ha) 회귀가 아닙니다.
좌표 없는 나무까지 포함한 기존 `AGB_ref_t_ha`를 그대로 기준으로 씁니다.

- 고정식: `AGB = b0 + b1 × mean_height_m + b2 × cover_2m` (피복률 0~1).
- GRSM 15개 → ORNL 8개, ORNL 8개 → GRSM 15개의 두 지역 분리 평가.
- 방법 A는 기존 `methods/A_DF_nested_OOF/plot_comparison.csv`의 바깥 GroupKFold OOF 예측을 읽습니다.
  새 모델의 지역 분리 평가와 A의 조사구 GroupKFold 평가는 분할 방식이 다릅니다.
- 새 자료 다운로드, 탐지 재실행, 설정 탐색, 정규화·변환, 음수 예측 보정은 없습니다.
- 23개는 앞선 실험에서 이미 본 자료입니다. **내부 비교이며 독립 검증이 아닙니다.**

순서: 설치 → Drive 마운트 → 입력 확인 → 특징 → 회귀 2방향 → 비교 → 그림 → 저장·ZIP 다운로드.
기존 입력과 결과는 읽기만 합니다. 이번 실험의 저장 경로가 이미 있으면 덮어쓰지 않고 중단합니다.


## 1. 설치
분석용 패키지만 설치합니다. 원본 자료를 받는 코드는 없습니다.


In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<4" "rasterio>=1.4,<2" "geopandas>=1.0,<2" "pyogrio>=0.9,<1" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "tabulate>=0.9,<1"


## 2. Drive 마운트와 고정 경로


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import importlib.metadata
import json
import shutil
import tempfile
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask, geometry_window
from rasterio.windows import Window
from shapely.geometry import mapping
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path('/content/drive/MyDrive/neon_grsm_ornl')
DRIVE_OUT = ROOT / 'outputs/area_chm_20261112'
ZIP_PATH = Path('/content/neon_area_chm_results.zip')
assert ROOT.is_dir(), f'Drive 원본 폴더 없음: {ROOT}'

# 입력 위치만 조정할 수 있습니다. 모델 설정은 탐색하지 않습니다.
RUN_DIR_OVERRIDE = ''  # 3차 결과 폴더. 기본: outputs/f55cc4636e94 자동 탐색
FOOTPRINT_OVERRIDE = ''  # 못 찾으면 업로드 후 /content/...gpkg 경로를 입력
METHOD_A_OVERRIDE = ''  # 필요할 때만 정확한 methods/A_DF_nested_OOF/plot_comparison.csv

def ensure_new_outputs():
    for path in [DRIVE_OUT, ZIP_PATH]:
        if path.exists():
            raise FileExistsError(
                f'기존 결과 보호로 중단: {path}. 이전 결과를 별도 위치에 보관한 후 다시 실행하세요.'
            )

ensure_new_outputs()
WORK = Path(tempfile.mkdtemp(prefix='neon_area_chm_', dir='/content'))
FIGURES = WORK / 'figures'
FIGURES.mkdir()
print('원본(읽기 전용):', ROOT)
print('이번 결과:', DRIVE_OUT, '\nZIP:', ZIP_PATH)


## 3. 입력 확인: 3차 자료·23개 조사구·파일 존재·좌표계

`outputs/f55cc4636e94`를 먼저 찾습니다. 그 위치에 없으면 `outputs` 아래에서 방법 A 경로를
검색해 후보를 보여줍니다. 여러 실행 폴더가 있으면 **3차 결과**를 선택하세요.
경계는 선택한 3차 실행 폴더에서 먼저 찾고, 없으면 `outputs` 아래 3차 이름 후보를 찾습니다.
계속 없으면 왼쪽 파일 패널에 기존 GPKG를 업로드하고 셀의 질문에 `/content/...gpkg` 경로를 입력합니다.
자동 다운로드는 하지 않습니다. 다른 차수 자료를 임의로 대체하지 마세요.

23개 ID는 로컬 3차 CSV에서 고정했습니다. CSV·경계·CHM을 모두 같은 ID로 연결하고
CHM은 지정된 2021년 `f55cc4636e94` 파일만 사용합니다.


In [ ]:
def choose_existing(candidates, prompt):
    candidates = sorted(set(Path(p) for p in candidates if Path(p).is_file()))
    if len(candidates) == 1:
        print(prompt, candidates[0])
        return candidates[0]
    if candidates:
        for i, path in enumerate(candidates, 1):
            print(f'[{i}] {path}')
        answer = input(prompt + ' 번호 또는 기존 파일의 전체 경로: ').strip()
        path = candidates[int(answer) - 1] if answer.isdigit() and 1 <= int(answer) <= len(candidates) else Path(answer)
    else:
        path = Path(input(prompt + ' 기존 파일을 업로드/배치한 전체 경로: ').strip())
    assert path.is_file(), f'입력 파일을 찾을 수 없습니다: {path}'
    return path

method_rel = Path('methods/A_DF_nested_OOF/plot_comparison.csv')
if METHOD_A_OVERRIDE:
    A_PATH = Path(METHOD_A_OVERRIDE)
elif RUN_DIR_OVERRIDE:
    A_PATH = Path(RUN_DIR_OVERRIDE) / method_rel
else:
    preferred = ROOT / 'outputs/f55cc4636e94' / method_rel
    candidates = [preferred] if preferred.is_file() else [
        p for p in (ROOT / 'outputs').rglob('plot_comparison.csv')
        if p.parent.name == 'A_DF_nested_OOF' and p.parent.parent.name == 'methods'
    ]
    A_PATH = choose_existing(candidates, '3차 결과의 방법 A CSV를 선택하세요.')
assert A_PATH.is_file(), f'방법 A CSV 없음: {A_PATH}'
assert A_PATH.parts[-3:] == ('methods', 'A_DF_nested_OOF', 'plot_comparison.csv'),     '요약/배포 결과가 아니라 methods/A_DF_nested_OOF/plot_comparison.csv를 지정하세요.'
RUN_DIR = A_PATH.parents[2]

if FOOTPRINT_OVERRIDE:
    FOOTPRINT_PATH = Path(FOOTPRINT_OVERRIDE)
elif (RUN_DIR / 'sampled_footprints.gpkg').is_file():
    FOOTPRINT_PATH = RUN_DIR / 'sampled_footprints.gpkg'
else:
    candidates = [p for p in (ROOT / 'outputs').rglob('sampled_footprints.gpkg')
                  if any(tag in str(p.parent).lower() for tag in ['f55cc4636e94', 'v03', 'v3'])]
    FOOTPRINT_PATH = choose_existing(candidates, '3차 조사구 경계 GPKG를 선택하세요.')
assert FOOTPRINT_PATH.is_file(), f'조사구 경계 없음: {FOOTPRINT_PATH}'

print('방법 A:', A_PATH)
print('경계:', FOOTPRINT_PATH)
print('주의: 방법 A는 기존 바깥 GroupKFold OOF; 새 방법은 지역 분리 평가입니다.')


In [ ]:
EXPECTED_IDS = ['GRSM_001', 'GRSM_009', 'GRSM_012', 'GRSM_014', 'GRSM_016', 'GRSM_017', 'GRSM_047', 'GRSM_049', 'GRSM_050', 'GRSM_051', 'GRSM_055', 'GRSM_057', 'GRSM_058', 'GRSM_059', 'GRSM_060', 'ORNL_001', 'ORNL_004', 'ORNL_006', 'ORNL_009', 'ORNL_010', 'ORNL_027', 'ORNL_049', 'ORNL_057']

EXPECTED_COUNTS = {'GRSM': 15, 'ORNL': 8}

def require_columns(frame, columns, label):
    missing = set(columns) - set(frame.columns)
    assert not missing, f'{label} 누락 열: {sorted(missing)}'

def assert_cohort(frame, label):
    require_columns(frame, ['plotID', 'siteID'], label)
    assert len(frame) == 23 and frame.plotID.is_unique, f'{label}: 고유 23행 필요'
    assert set(frame.plotID) == set(EXPECTED_IDS), f'{label}: 3차 23개 ID와 불일치'
    assert frame.siteID.value_counts().to_dict() == EXPECTED_COUNTS, f'{label}: GRSM 15 + ORNL 8 필요'
    assert (frame.plotID.str.split('_').str[0] == frame.siteID).all(), f'{label}: siteID 불일치'

reference = pd.read_csv(A_PATH, dtype={'plotID': str, 'siteID': str})
require_columns(reference, ['AGB_ref_t_ha', 'AGB_est_t_ha', 'C_ref_tC_ha'], '방법 A CSV')
assert_cohort(reference, '방법 A CSV')
for col in ['AGB_ref_t_ha', 'AGB_est_t_ha', 'C_ref_tC_ha']:
    reference[col] = pd.to_numeric(reference[col], errors='raise')
    assert np.isfinite(reference[col]).all(), f'{col}: 결측/무한값. 조사구 제외 없이 입력을 확인하세요.'
for col in ['field_year', 'imagery_year']:
    if col in reference:
        assert (reference[col] == 2021).all(), f'{col}: 2021 이외 자료'

carbon_delta = reference.AGB_ref_t_ha * 0.5 - reference.C_ref_tC_ha
carbon_check = bool(np.allclose(carbon_delta, 0, rtol=1e-7, atol=1e-8))
print('확인만 출력: AGB_ref × 0.5 = C_ref:', carbon_check,
      '| 최대 절대 차이:', float(carbon_delta.abs().max()))
print('분석 기준은 좌표 없는 나무도 포함한 AGB_ref_t_ha입니다. 탄소로 환산하지 않습니다.')

# 자동 manifest가 있으면 지정된 23개 CHM 경로와 일치하는지도 확인합니다.
CHM_PATHS = {pid: ROOT / 'data/crops' / f'{pid}_2021_chm_f55cc4636e94.tif' for pid in EXPECTED_IDS}
MANIFEST_PATH = RUN_DIR / 'raster_manifest.csv'
if MANIFEST_PATH.is_file():
    manifest = pd.read_csv(MANIFEST_PATH, dtype={'plotID': str})
    require_columns(manifest, ['plotID', 'chm'], 'raster_manifest')
    assert manifest.plotID.is_unique and set(manifest.plotID) == set(EXPECTED_IDS)
    for row in manifest.itertuples():
        assert Path(row.chm).name == CHM_PATHS[row.plotID].name, f'manifest CHM 불일치: {row.plotID}'
else:
    print('manifest 없음: 지정한 고정 CHM 파일명과 파일 존재를 직접 확인합니다.')
missing_files = [str(p) for p in CHM_PATHS.values() if not p.is_file()]
assert not missing_files, '기존 CHM이 없습니다. 자동 다운로드하지 않습니다:\n' + '\n'.join(missing_files)

layers = gpd.list_layers(FOOTPRINT_PATH)
if len(layers) == 1:
    layer_name = layers.iloc[0]['name']
else:
    display(layers)
    layer_name = input('조사구 경계가 든 레이어 이름: ').strip()
    assert layer_name in set(layers['name'])
footprints = gpd.read_file(FOOTPRINT_PATH, layer=layer_name)
require_columns(footprints, ['plotID', 'geometry'], '경계 GPKG')
assert footprints.crs is not None, '경계 CRS 누락. 임의 CRS를 부여하지 않습니다.'
assert len(footprints) == 23 and footprints.plotID.is_unique, '경계는 조사구당 1행, 총 23행이어야 합니다.'
assert set(footprints.plotID) == set(EXPECTED_IDS)
assert footprints.geometry.notna().all() and (~footprints.geometry.is_empty).all()
assert footprints.geometry.is_valid.all(), '유효하지 않은 경계. 자동 수정 없이 원본을 확인하세요.'
assert footprints.geom_type.isin(['Polygon', 'MultiPolygon']).all()
if 'siteID' in footprints:
    assert_cohort(footprints, '경계 GPKG')

audit_rows = []
for pid, path in CHM_PATHS.items():
    with rasterio.open(path) as src:
        assert src.crs is not None and src.crs.to_epsg() in (32617, 32616), f'{pid}: 예상 밖 CHM CRS {src.crs}'
        assert src.count == 1, f'{pid}: 단일 CHM 밴드 필요'
        assert np.allclose(src.res, [1, 1], rtol=0, atol=1e-6), f'{pid}: 1m CHM 필요 {src.res}'
        assert abs(src.transform.b) < 1e-10 and abs(src.transform.d) < 1e-10, '회전 래스터는 원본 확인 필요'
        geom = footprints.loc[footprints.plotID == pid].to_crs(src.crs).geometry.iloc[0]
        assert np.isfinite(geom.area) and geom.area > 0, f'{pid}: 면적 오류'
        audit_rows.append(dict(plotID=pid, chm_crs=str(src.crs), resolution_m=src.res[0],
                               dtype=src.dtypes[0], source_nodata=src.nodata,
                               polygon_area_m2=float(geom.area), chm_path=str(path)))
input_audit = pd.DataFrame(audit_rows)
print('경계 원본 CRS:', footprints.crs, '| 각 조사구는 해당 CHM CRS로 재투영합니다.')
display(input_audit)
display(reference.groupby('siteID').size().rename('n').to_frame())


## 4. 조사구별 특징 (픽셀 중심 포함, all_touched=False)

`n_pixels`는 경계 안에 중심이 있는 전체 1m 픽셀 수(결측 포함), `n_valid_pixels`는 유효 픽셀 수입니다.
`n_nodata`는 그중 nodata/NaN/비유한값/래스터 밖 픽셀 수, `nodata_frac = n_nodata / n_pixels`입니다.
평균 높이와 피복률의 분모는 **유효 픽셀**이며, 0m도 유효값으로 평균에 포함합니다.
`n_zero`는 유효한 0m 픽셀만 셉니다. 결측률이 10%를 넘으면 경고 열만 남기고 제외하지 않습니다.
픽셀/유효 픽셀이 아예 없으면 모델을 만들 수 없으므로 전체 실행을 중단합니다. 임의 제외나 대체는 없습니다.

정수 CHM도 float32로 읽습니다. `merge`/`mask`를 쓰지 않고 창 읽기와 geometry mask를 분리하며,
원본 nodata가 None일 때 창 밖 채움값을 명시적으로 NaN으로 지정합니다. 0으로 채우지 않습니다.


In [ ]:
def extract_features(pid):
    path = CHM_PATHS[pid]
    with rasterio.open(path) as src:
        geom = footprints.loc[footprints.plotID == pid].to_crs(src.crs).geometry.iloc[0]
        # boundless 창: 경계가 영상 밖으로 나가도 해당 픽셀을 결측으로 기록합니다.
        window = geometry_window(src, [mapping(geom)], boundless=True)
        # 정수 원본에 NaN을 채우지 않습니다. float32 배열을 먼저 만들고 영상 내부만 읽습니다.
        fill_nodata = src.nodata if src.nodata is not None else np.nan
        shape = (int(window.height), int(window.width))
        heights = np.full(shape, fill_nodata, dtype='float32')
        source_valid = np.zeros(shape, dtype=bool)
        c0, r0 = max(0, int(window.col_off)), max(0, int(window.row_off))
        c1 = min(src.width, int(window.col_off + window.width))
        r1 = min(src.height, int(window.row_off + window.height))
        if c1 > c0 and r1 > r0:
            read_window = Window(c0, r0, c1 - c0, r1 - r0)
            rs = slice(r0 - int(window.row_off), r1 - int(window.row_off))
            cs = slice(c0 - int(window.col_off), c1 - int(window.col_off))
            heights[rs, cs] = src.read(1, window=read_window, masked=False, out_dtype='float32')
            source_valid[rs, cs] = src.read_masks(1, window=read_window) > 0
        inside = geometry_mask([mapping(geom)], out_shape=heights.shape,
                               transform=src.window_transform(window),
                               all_touched=False, invert=True)
        valid = inside & source_valid & np.isfinite(heights)
        # 명시된 nodata도 확인합니다(원본의 nodata 메타데이터 우선).
        if src.nodata is not None and np.isfinite(src.nodata):
            valid &= heights != np.float32(src.nodata)
        n_pixels = int(inside.sum())
        n_valid = int(valid.sum())
        assert n_pixels > 0 and n_valid > 0, f'{pid}: 사용 가능한 픽셀 없음. 23개 전체 실행을 중단합니다.'
        values = heights[valid]
        n_nodata = n_pixels - n_valid
        return dict(plotID=pid, polygon_area_m2=float(geom.area), n_pixels=n_pixels,
                    n_nodata=n_nodata, nodata_frac=n_nodata / n_pixels,
                    n_zero=int(np.count_nonzero(values == 0)),
                    mean_height_m=float(values.mean(dtype=np.float64)),
                    cover_2m=float(np.count_nonzero(values >= 2) / n_valid),
                    n_valid_pixels=n_valid, nodata_warning_gt10pct=n_nodata / n_pixels > 0.10)

feature_rows = [extract_features(pid) for pid in EXPECTED_IDS]
features = pd.DataFrame(feature_rows).merge(
    reference[['plotID', 'siteID', 'AGB_ref_t_ha', 'AGB_est_t_ha']].rename(
        columns={'AGB_ref_t_ha': 'AGB_ref_Mg_ha', 'AGB_est_t_ha': 'AGB_A_Mg_ha'}),
    on='plotID', validate='one_to_one')
FEATURE_COLUMNS = ['plotID', 'siteID', 'polygon_area_m2', 'n_pixels', 'n_nodata',
                   'nodata_frac', 'n_zero', 'mean_height_m', 'cover_2m',
                   'AGB_ref_Mg_ha', 'AGB_A_Mg_ha', 'n_valid_pixels', 'nodata_warning_gt10pct']
features = features[FEATURE_COLUMNS].sort_values('plotID').reset_index(drop=True)
assert_cohort(features, 'features')
assert (features.n_pixels == features.n_valid_pixels + features.n_nodata).all()
assert (features.n_zero <= features.n_valid_pixels).all()
assert features.cover_2m.between(0, 1).all()
assert np.isfinite(features[['mean_height_m', 'cover_2m', 'AGB_ref_Mg_ha', 'AGB_A_Mg_ha']]).all().all()
print('결측률 >10% 경고 조사구:', int(features.nodata_warning_gt10pct.sum()), '| 제외한 조사구: 0')
display(features)


## 5. 고정 선형 회귀 — 지역 분리 2방향

학습 지역의 기준 AGB만 `fit`에 전달합니다. 평가 지역 정답은 학습·설정 선택에 쓰지 않습니다.
정규화·변환·규제·변수 선택이 없고 절편을 포함합니다. 음수 예측은 그대로 두고 표시합니다.
학습 지역 안 적합(in-sample)은 참고용으로 별도 저장하며 비교 결론에 사용하지 않습니다.


In [ ]:
X_COLUMNS = ['mean_height_m', 'cover_2m']
DIRECTIONS = [('D1_GRSM_to_ORNL', 'GRSM', 'ORNL'), ('D2_ORNL_to_GRSM', 'ORNL', 'GRSM')]
prediction_parts, in_sample_parts, coefficient_rows = [], [], []

for direction, train_site, test_site in DIRECTIONS:
    train = features.loc[features.siteID == train_site].copy()
    test = features.loc[features.siteID == test_site].copy()
    assert set(train.plotID).isdisjoint(test.plotID)
    assert len(train) == EXPECTED_COUNTS[train_site] and len(test) == EXPECTED_COUNTS[test_site]
    model = LinearRegression(fit_intercept=True, positive=False)
    model.fit(train[X_COLUMNS], train.AGB_ref_Mg_ha)
    test_pred = model.predict(test[X_COLUMNS])  # 평가 지역 정답을 전달하지 않습니다.
    train_pred = model.predict(train[X_COLUMNS])
    coefficient_rows.append(dict(direction=direction, train_site=train_site, test_site=test_site,
        b0=float(model.intercept_), b1=float(model.coef_[0]), b2=float(model.coef_[1]),
        train_n=len(train), train_R2=float(model.score(train[X_COLUMNS], train.AGB_ref_Mg_ha))))

    out = test[['plotID', 'siteID']].copy()
    out['direction'], out['train_site'], out['test_site'] = direction, train_site, test_site
    out['AGB_ref'] = test.AGB_ref_Mg_ha
    out['AGB_area_pred'] = test_pred
    out['AGB_A_pred'] = test.AGB_A_Mg_ha
    out['error_area'] = out.AGB_area_pred - out.AGB_ref
    out['error_A'] = out.AGB_A_pred - out.AGB_ref
    out['negative_flag'] = out.AGB_area_pred < 0
    prediction_parts.append(out)

    fitted = train[['plotID', 'siteID', 'AGB_ref_Mg_ha']].rename(columns={'AGB_ref_Mg_ha': 'AGB_ref'}).copy()
    fitted['AGB_area_pred'] = train_pred
    fitted['direction'], fitted['train_site'], fitted['test_site'] = direction, train_site, test_site
    in_sample_parts.append(fitted)

predictions = pd.concat(prediction_parts, ignore_index=True)
coefficients = pd.DataFrame(coefficient_rows)
in_sample = pd.concat(in_sample_parts, ignore_index=True)
assert_cohort(predictions, '평가 predictions')
assert (predictions.siteID == predictions.test_site).all()
assert (predictions.train_site != predictions.test_site).all()
assert np.isfinite(predictions[['AGB_ref', 'AGB_area_pred', 'AGB_A_pred']]).all().all()
assert len(coefficients) == 2
print('평가 예측: 23개, 조사구당 1회. 음수도 그대로 보존:', int(predictions.negative_flag.sum()))
display(coefficients)
display(predictions)


## 6. 방법 A와 같은 평가 조사구에서 비교

평균 편향은 `mean(추정−기준)`, 비율은 `평균 추정 ÷ 평균 기준`입니다(조사구별 비율의 평균 아님).
RMSE·편향·MAE는 Mg/ha, Pearson r과 비율은 무차원입니다. 상수 벡터의 r 또는 평균 기준이 0인
경우의 비율은 NaN으로 기록합니다. 지표 계산 시 조사구를 버리지 않습니다.

통합 23개 행은 **각 조사구의 지역이 평가일 때의 예측만** 합칩니다. 학습 적합도와 섞지 않습니다.
방법 A는 이번 지역 분리 재학습 결과가 아니라 기존 바깥 GroupKFold OOF 결과입니다.


In [ ]:
def metrics(ref, pred):
    y, p = np.asarray(ref, dtype=float), np.asarray(pred, dtype=float)
    assert len(y) == len(p) and len(y) > 0
    assert np.isfinite(y).all() and np.isfinite(p).all(), '지표 계산에서 결측 행을 제거하지 않습니다.'
    error = p - y
    r = float(np.corrcoef(y, p)[0, 1]) if np.std(y) > 0 and np.std(p) > 0 else np.nan
    return dict(n=len(y), RMSE_Mg_ha=float(np.sqrt(np.mean(error ** 2))),
                bias_Mg_ha=float(np.mean(error)), MAE_Mg_ha=float(np.mean(np.abs(error))),
                mean_estimate_over_mean_reference=float(p.mean() / y.mean()) if y.mean() != 0 else np.nan,
                pearson_r=r)

comparison_rows = []
for direction, train_site, test_site in DIRECTIONS:
    q = predictions.loc[predictions.direction == direction]
    for method, column in [('A_DF_nested_OOF', 'AGB_A_pred'), ('Area_CHM_linear', 'AGB_area_pred')]:
        comparison_rows.append(dict(method=method, evaluation_scope=f'{test_site} 평가',
            split_type='evaluation', direction=direction, train_site=train_site, test_site=test_site,
            **metrics(q.AGB_ref, q[column])))
for method, column in [('A_DF_nested_OOF', 'AGB_A_pred'), ('Area_CHM_linear', 'AGB_area_pred')]:
    comparison_rows.append(dict(method=method, evaluation_scope='통합 23개 평가',
        split_type='evaluation', direction='D1+D2', train_site='opposite_site_per_plot',
        test_site='GRSM+ORNL', **metrics(predictions.AGB_ref, predictions[column])))

# 참고 행은 새 모델의 학습 적합도입니다. A의 기존 OOF를 in-sample로 부르지 않습니다.
for direction, train_site, test_site in DIRECTIONS:
    q = in_sample.loc[in_sample.direction == direction]
    comparison_rows.append(dict(method='Area_CHM_linear', evaluation_scope=f'{train_site} 학습 지역 안 적합 (in-sample, 참고)',
        split_type='in_sample_reference_only', direction=direction,
        train_site=train_site, test_site=test_site, **metrics(q.AGB_ref, q.AGB_area_pred)))
comparison = pd.DataFrame(comparison_rows)
evaluation = comparison.loc[comparison.split_type == 'evaluation'].copy()
assert len(evaluation) == 6 and len(comparison) == 8
assert evaluation.n.tolist() == [8, 8, 15, 15, 23, 23]
display(evaluation)
print('아래 학습 지역 안 적합(in-sample)은 참고용이며 비교 결론에 쓰지 않습니다.')
display(comparison.loc[comparison.split_type == 'in_sample_reference_only'])


## 7. 그림 — 평가 예측만, 음수도 축 범위에 포함


In [ ]:
COLORS = {'GRSM': 'tab:blue', 'ORNL': 'tab:orange'}
fig, ax = plt.subplots(figsize=(7, 7))
for site, q in predictions.groupby('siteID'):
    ax.scatter(q.AGB_ref, q.AGB_area_pred, marker='o', color=COLORS[site],
               s=55, alpha=0.8, label=f'{site}: Area CHM (site holdout)')
    ax.scatter(q.AGB_ref, q.AGB_A_pred, marker='^', facecolors='none', edgecolors=COLORS[site],
               s=65, alpha=0.9, label=f'{site}: Method A (existing OOF)')
all_values = predictions[['AGB_ref', 'AGB_area_pred', 'AGB_A_pred']].to_numpy()
lo, hi = min(0.0, float(all_values.min())), max(1.0, float(all_values.max()))
margin = 0.05 * (hi - lo)
limits = [lo - margin, hi + margin]
ax.plot(limits, limits, 'k--', linewidth=1, label='1:1')
ax.set(xlim=limits, ylim=limits, xlabel='Reference AGB (Mg/ha)', ylabel='Predicted AGB (Mg/ha)',
       title='23 evaluation plots: Area CHM vs Method A')
ax.set_aspect('equal', adjustable='box')
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / 'scatter_area_vs_A.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, column, label in zip(axes, X_COLUMNS, ['Mean CHM height (m; zeros included)', 'Cover >= 2 m (fraction)']):
    for site, q in features.groupby('siteID'):
        ax.scatter(q[column], q.AGB_ref_Mg_ha, color=COLORS[site], label=site, s=45, alpha=0.8)
    ax.set(xlabel=label, ylabel='Reference AGB (Mg/ha)')
    ax.grid(alpha=0.2)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'features.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)


## 8. 저장·ZIP 다운로드와 마지막 비교표

4개 CSV, 그림 2개, `report.md`를 ZIP으로 묶고 Drive `outputs/area_chm_20261112/`에도
각 파일과 ZIP 사본을 저장합니다. 기존 저장 경로가 있으면 덮어쓰지 않습니다.
보고서 맨 위 결과는 실행 후 수치로 작성되며, 다음 단계 판단란은 빈칸으로 둡니다.


In [ ]:
ensure_new_outputs()
features.to_csv(WORK / 'features.csv', index=False)
predictions.to_csv(WORK / 'predictions.csv', index=False)
comparison.to_csv(WORK / 'comparison.csv', index=False)
coefficients.to_csv(WORK / 'coefficients.csv', index=False)

area_all = evaluation.loc[(evaluation.method == 'Area_CHM_linear') & (evaluation.direction == 'D1+D2')].iloc[0]
a_all = evaluation.loc[(evaluation.method == 'A_DF_nested_OOF') & (evaluation.direction == 'D1+D2')].iloc[0]
headline = (f"내부 비교 23개: 새 방법 / 방법 A의 RMSE {area_all.RMSE_Mg_ha:.2f} / {a_all.RMSE_Mg_ha:.2f} Mg/ha, "
            f"평균 편향 {area_all.bias_Mg_ha:+.2f} / {a_all.bias_Mg_ha:+.2f} Mg/ha, "
            f"평균 추정÷평균 기준 {area_all.mean_estimate_over_mean_reference:.3f} / {a_all.mean_estimate_over_mean_reference:.3f} (독립 검증 아님).")

def readable_table(frame, compact=False):
    columns = ['evaluation_scope', 'method', 'n', 'RMSE_Mg_ha', 'bias_Mg_ha']
    if not compact:
        columns += ['MAE_Mg_ha']
    columns += ['mean_estimate_over_mean_reference']
    if not compact:
        columns += ['pearson_r']
    return frame[columns].replace({'method': {
        'A_DF_nested_OOF': '방법 A (기존 OOF)', 'Area_CHM_linear': '새 방법 (조사구 CHM)'}}).rename(columns={
        'evaluation_scope': '평가 범위', 'method': '방법', 'n': 'n', 'RMSE_Mg_ha': 'RMSE (Mg/ha)',
        'bias_Mg_ha': '평균 편향 (Mg/ha)', 'MAE_Mg_ha': 'MAE (Mg/ha)',
        'mean_estimate_over_mean_reference': '평균 추정÷평균 기준', 'pearson_r': 'Pearson r'})

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

versions = {name: importlib.metadata.version(name) for name in
            ['numpy', 'pandas', 'geopandas', 'rasterio', 'scikit-learn', 'matplotlib']}
report = headline + '\n\n' + '# NEON 조사구 CHM AGB 회귀 — 2026-11-12\n\n'
report += '## 평가 비교\n\n' + readable_table(evaluation).to_markdown(index=False, floatfmt='.3f') + '\n\n'
report += '''단위는 지상부 AGB Mg/ha (= t/ha)입니다. 기준은 좌표 없는 나무까지 포함한 AGB_ref_t_ha입니다.
편향 = mean(추정−기준), 비율 = mean(추정)/mean(기준). 통합 23개는 지역이 평가일 때의 예측만 사용했습니다.
방법 A는 기존 바깥 GroupKFold OOF 예측이므로 새 방법의 지역 분리 평가와 평가 분할 방식이 다릅니다.
따라서 이 표는 같은 조사구에 대한 내부 비교이며, 두 방법의 동일한 지역 외 일반화 성능 비교는 아닙니다.

## 학습 지역 안 적합 (in-sample, 참고용)

아래 행은 새 모델이 학습에 사용한 지역에서의 적합도이며 비교 결론에 사용하지 않습니다.

'''
report += readable_table(comparison.loc[comparison.split_type == 'in_sample_reference_only']).to_markdown(index=False, floatfmt='.3f') + '\n\n'
report += '## 계수와 학습 R²\n\n' + coefficients.to_markdown(index=False, floatfmt='.6f') + '\n\n'
report += '''고정식: AGB = b0 + b1 × mean_height_m + b2 × cover_2m (0~1).
절편 포함, 정규화·변환·규제·설정 탐색 없음. 평가 지역 정답은 학습에 사용하지 않았습니다.
음수 예측은 보정하지 않았고 predictions.csv의 negative_flag로 표시했습니다.

## 특징 및 입력 점검

- 경계와 CHM의 좌표계를 맞추고 픽셀 중심이 경계 안에 있는 1m 픽셀만 사용(all_touched=False).
- n_pixels는 결측을 포함한 전체 경계 내부 픽셀, n_valid_pixels는 유효 픽셀 수입니다.
- mean_height_m은 유효 픽셀의 평균(0m 포함); cover_2m은 유효 픽셀 중 CHM ≥ 2m 비율입니다.
- 결측률 >10%는 nodata_warning_gt10pct에만 표시하며, 조사구를 제외하지 않았습니다.
- 정수 CHM도 float32로 읽고 nodata=None이면 창 밖 채움값을 NaN으로 명시했습니다.

'''
report += f'- 결측률 >10%: {int(features.nodata_warning_gt10pct.sum())}개; 음수 평가 예측: {int(predictions.negative_flag.sum())}개; 제외: 0개.\n'
report += f'- AGB_ref × 0.5 = C_ref 확인만: {carbon_check}; 최대 절대 차이 {float(carbon_delta.abs().max()):.8g}.\n'
report += f'- 방법 A 입력: `{A_PATH}` (SHA-256 `{sha256_file(A_PATH)}`).\n'
report += f'- 경계 입력: `{FOOTPRINT_PATH}`, 레이어 `{layer_name}` (SHA-256 `{sha256_file(FOOTPRINT_PATH)}`).\n'
report += f'- 경계 원본 CRS: `{footprints.crs}`; 아래 각 CHM CRS에서 폴리곤 면적 계산.\n\n'
report += input_audit.to_markdown(index=False) + '\n\n'
report += f'실행 시각(UTC): {datetime.now(timezone.utc).isoformat()}\n\n패키지 버전: `{json.dumps(versions)}`\n\n'
report += '''## 해석상 한계

- 23개는 앞선 실험에서 이미 본 자료로 내부 비교이며 독립 검증이 아닙니다.
- 방법 A의 설정(score 0.2, patch 600)도 같은 23개에서 선택되었습니다. 여기서는 배포 설정으로 다시 예측하지 않고 기존 바깥 GroupKFold OOF 열을 사용했습니다.
- 학습 표본은 8~15개로 작고 지역은 2곳뿐입니다.
- NEON LiDAR CHM과 국내 Meta 추정 높이는 다른 자료이므로 국내 성능으로 해석할 수 없습니다.
- 평균 추정÷평균 기준이 1에 가까워도 조사구별 오차와 RMSE·MAE·편향·상관을 함께 봐야 합니다.
- 방법 A와 새 방법의 평가 분할이 다릅니다. 비교표만으로 우열이나 지역 외 성능을 확정할 수 없습니다.
- 기준 AGB도 기존 실측 기반 공식의 산출값이며 벌채 실측의 독립 정답은 아닙니다.

## 다음 단계 판단 (직접 작성)

- 추가 검증 진행 여부: ____________________
- 판단 근거: ____________________
- 다음 자료·실험: ____________________
'''
(WORK / 'report.md').write_text(report, encoding='utf-8')

expected_files = ['features.csv', 'predictions.csv', 'comparison.csv', 'coefficients.csv',
                  'figures/scatter_area_vs_A.png', 'figures/features.png', 'report.md']
assert all((WORK / name).is_file() for name in expected_files)
assert len(pd.read_csv(WORK / 'features.csv')) == 23
assert len(pd.read_csv(WORK / 'predictions.csv')) == 23
assert len(pd.read_csv(WORK / 'comparison.csv')) == 8
assert len(pd.read_csv(WORK / 'coefficients.csv')) == 2
with zipfile.ZipFile(ZIP_PATH, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in expected_files:
        archive.write(WORK / name, arcname=name)
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert set(archive.namelist()) == set(expected_files) and archive.testzip() is None

DRIVE_OUT.mkdir(parents=True, exist_ok=False)
for name in expected_files:
    target = DRIVE_OUT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    with (WORK / name).open('rb') as source, target.open('xb') as dest:
        shutil.copyfileobj(source, dest)
with ZIP_PATH.open('rb') as source, (DRIVE_OUT / ZIP_PATH.name).open('xb') as dest:
    shutil.copyfileobj(source, dest)
assert sha256_file(ZIP_PATH) == sha256_file(DRIVE_OUT / ZIP_PATH.name)
print('저장 완료:', DRIVE_OUT, '\nZIP:', ZIP_PATH)
files.download(str(ZIP_PATH))

# 마지막 출력: 사람이 읽는 평가 비교표. 학습 적합도는 이 표에 섞지 않습니다.
display(Markdown('**방법 A vs 새 방법 — 같은 평가 조사구, AGB (Mg/ha)**'))
display(Markdown(readable_table(evaluation, compact=True).to_markdown(index=False, floatfmt='.3f')))
